# Customer Segmentation using RFM Analysis

This notebook segments customers based on purchasing behavior using RFM analysis.

Key goals:
- Identify high-value and low-value customers
- Understand customer behavior patterns
- Estimate customer lifetime value (CLTV proxy)
- Provide targeted strategies for each segment

RFM is a customer segmentation technique based on:

- Recency → How recently a customer purchased
- Frequency → How often they purchase
- Monetary → How much they spend

## Data Preparation

We calculate RFM metrics per customer.

In [ ]:
import pandas as pd
import numpy as np

# Convert date
orders['order_date'] = pd.to_datetime(orders['order_date'])

snapshot_date = orders['order_date'].max() + pd.Timedelta(days=1)

rfm = orders.groupby('customer_id').agg({
    'order_date': lambda x: (snapshot_date - x.max()).days,  # Recency
    'order_id': 'count',                                    # Frequency
    'revenue': 'sum'                                        # Monetary
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.reset_index(inplace=True)

## RFM Scoring

Customers are scored from 1 to 5:

- Recency → lower is better (recent buyers)
- Frequency → higher is better
- Monetary → higher is better

In [ ]:
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

rfm['RFM_score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

## Customer Segments

We define segments based on RFM scores:

- Champions → High R, F, M
- Loyal Customers → High F, good M
- At Risk → Low R but high past value
- Lost Customers → Low R, F, M

In [ ]:
def segment_customer(row):
    if row['R_score'] == 5 and row['F_score'] >= 4:
        return 'Champions'
    elif row['F_score'] >= 4:
        return 'Loyal Customers'
    elif row['R_score'] <= 2 and row['F_score'] >= 3:
        return 'At Risk'
    else:
        return 'Lost Customers'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

## Segment Distribution

Understanding % of customers in each segment.

In [ ]:
segment_dist = rfm['Segment'].value_counts(normalize=True) * 100
segment_dist

## Revenue Contribution Analysis

We analyze how much revenue is generated by top customers.

In [ ]:
rfm_sorted = rfm.sort_values(by='Monetary', ascending=False)

top_20 = int(0.2 * len(rfm_sorted))
top_customers = rfm_sorted.head(top_20)

top_revenue_share = top_customers['Monetary'].sum() / rfm['Monetary'].sum()
top_revenue_share

## Segment Analysis

We analyze behavior across segments.

In [ ]:
segment_analysis = rfm.groupby('Segment').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean'
})

segment_analysis

## CLTV Analysis (Proxy)

We estimate customer value using Monetary and Frequency.

In [ ]:
rfm['CLTV_proxy'] = rfm['Monetary'] * rfm['Frequency']

cltv_analysis = rfm.groupby('Segment')['CLTV_proxy'].mean()
cltv_analysis

## Key Insights

- Small % of customers generate majority of revenue
- Champions contribute disproportionately high value
- Large number of customers fall into low-value segments
- At-risk customers represent recovery opportunity

## Segment-wise Strategy

### Champions
- Reward with loyalty programs
- Early access to products

### Loyal Customers
- Upsell and cross-sell
- Personalized recommendations

### At Risk
- Targeted re-engagement campaigns
- Limited-time offers

### Lost Customers
- Win-back campaigns
- Feedback collection

## Business Impact

- Retaining top 20% customers is critical for revenue stability
- Improving at-risk segment can significantly boost revenue
- Focus should shift from acquisition to retention